# Classical ML Failure Modes — AI Lab Instructor Notebook

*Classical ML · Medium*

Learn the failure modes that actually break ML in production and in interviews: leakage, bias, spurious correlations, dataset shift, and improper validation.

**Outcome.** Students can detect leakage, design correct splits, debug spurious correlations, and build robust baselines.

6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.linear_model import LogisticRegression

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Classical ML Failure Modes — Student Lab (Titanic)

This lab focuses on failure modes: leakage, spurious correlations, and bad validation.

## Section 0 — Load Titanic (Kaggle) with fallback

Expected path: `data/titanic/train.csv`

If missing, we generate a tiny synthetic dataset so notebook still runs.

In [ ]:
$10

**Why this works.** # Classical ML Failure Modes — Instructor (Solutions + Rationale)

**Spurious correlation** is when your model finds a statistical relationship between two variables, but it’s not actually causal—it’s just a coincidence. For instance, you might see that ice cream sales and shark attacks both rise in the summer, but one doesn’t cause the other—they’re just both linked to warmer weather.

**Slice analysis,**  is a way of breaking down your model's performance across different "slices" or subgroups of data. For example, you test accuracy separately for different genders, age groups, or geographies—this helps you spot hidden biases or weaknesses in the model that might not appear when you look at the overall performance. So, basically, slice analysis is a deeper dive to ensure your model is working well for all kinds of subpopulations, not just an overall average

## Task 2: Minimal baseline features
*Section: Baseline + Proper Validation*

## Section 1 — Baseline + Proper Validation

### Task 1.1: Minimal baseline features

Use only: Pclass, Sex, Age, Fare (simple).

- Create X/y
- Handle missing Age/Fare
- One-hot encode Sex

**Checkpoint:** Why do we start with a minimal baseline?

In [ ]:
y = df['Survived'].astype(int).values
X = df[['Pclass','Sex','Age','Fare']].copy()
X['Age'] = X['Age'].fillna(X['Age'].median())
X['Fare'] = X['Fare'].fillna(X['Fare'].median())
X = pd.get_dummies(X, columns=['Sex'], drop_first=True)

Xtr, Xva, ytr, yva = train_test_split(X.values, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=2000)
clf.fit(Xtr, ytr)
pred = clf.predict(Xva)
proba = clf.predict_proba(Xva)[:,1]
print('baseline acc', accuracy_score(yva, pred))
print('baseline auc', roc_auc_score(yva, proba))
check('baseline_acc_reasonable', accuracy_score(yva, pred) > 0.5)
check('baseline_auc_reasonable', roc_auc_score(yva, proba) > 0.5)

# Rationale: start simple to create a reference point; complex models without baselines are not interpretable.

In [ ]:
# Checks
check('baseline_acc_reasonable', accuracy_score(yva, pred) > 0.5)
check('baseline_auc_reasonable', roc_auc_score(yva, proba) > 0.5)

**Why this works.** start simple to create a reference point; complex models without baselines are not interpretable.

## Task 3: Create a leaky feature
*Section: Leakage*

## Section 2 — Leakage

### Task 2.1: Create a leaky feature

Intentionally create a feature that encodes the label (e.g., `leak = Survived`).
Train again and observe metric inflation.

**Checkpoint:** How can you detect leakage quickly in an interview?

In [ ]:
# Leakage demo
X_leak = X.copy()
X_leak['leak'] = y
Xtr, Xva, ytr, yva = train_test_split(X_leak.values, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=2000)
clf.fit(Xtr, ytr)
acc_with_leak = accuracy_score(yva, clf.predict(Xva))
print('acc_with_leak', acc_with_leak)
check('leak_inflated', acc_with_leak > 0.95)

# Rationale: leakage produces unrealistically high validation. Detect by inspecting features and using time-aware splits.

In [ ]:
# Checks
check('leak_inflated', acc_with_leak > 0.95)

**Why this works.** leakage produces unrealistically high validation. Detect by inspecting features and using time-aware splits.

## Task 4: Evaluate slices
*Section: Spurious correlations + slice analysis*

## Section 3 — Spurious correlations + slice analysis

### Task 3.1: Evaluate slices

Compute accuracy by groups (Sex, Pclass).

- Make predictions on validation
- Report metrics by slice

**Interview Angle:** Why can overall accuracy hide severe subgroup failures?

In [ ]:
# Slice analysis
idx = np.arange(len(df))
tr_idx, va_idx = train_test_split(idx, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=2000)
clf.fit(X.iloc[tr_idx].values, y[tr_idx])
pred = clf.predict(X.iloc[va_idx].values)
va_df = df.iloc[va_idx].copy()
va_df['pred'] = pred
va_df['correct'] = (va_df['pred'].values == va_df['Survived'].values).astype(int)
print('overall', va_df['correct'].mean())
print('by sex')
print(va_df.groupby('Sex')['correct'].mean())
print('by pclass')
print(va_df.groupby('Pclass')['correct'].mean())
check('slices_computed', len(va_df.groupby('Sex')['correct'].mean()) == 2)
check('overall_above_chance', va_df['correct'].mean() > 0.5)

# Rationale: aggregate metrics hide subgroup failures; always slice by key cohorts.

In [ ]:
# Checks
check('slices_computed', len(va_df.groupby('Sex')['correct'].mean()) == 2)
check('overall_above_chance', va_df['correct'].mean() > 0.5)

**Why this works.** aggregate metrics hide subgroup failures; always slice by key cohorts.

**Spurious correlation** is when your model finds a statistical relationship between two variables, but it’s not actually causal—it’s just a coincidence. For instance, you might see that ice cream sales and shark attacks both rise in the summer, but one doesn’t cause the other—they’re just both linked to warmer weather.

**Slice analysis,**  is a way of breaking down your model's performance across different "slices" or subgroups of data. For example, you test accuracy separately for different genders, age groups, or geographies—this helps you spot hidden biases or weaknesses in the model that might not appear when you look at the overall performance. So, basically, slice analysis is a deeper dive to ensure your model is working well for all kinds of subpopulations, not just an overall average

## Task 5: Simulate a shift in Fare distribution
*Section: Dataset shift (toy simulation)*

## Section 4 — Dataset shift (toy simulation)

### Task 4.1: Simulate a shift in Fare distribution

Create a shifted validation set by multiplying Fare and see how performance changes.

**Checkpoint:** How would you monitor drift in production?

In [ ]:
# Drift simulation
X_shift = X.copy()
if 'Fare' in X_shift.columns:
    X_shift['Fare'] = X_shift['Fare'] * 3.0
pred_shift = clf.predict(X_shift.iloc[va_idx].values)
acc_original = accuracy_score(y[va_idx], pred)
acc_shift = accuracy_score(y[va_idx], pred_shift)
print('acc_original', acc_original)
print('acc_shift', acc_shift)
check('shift_changes_acc', abs(acc_shift - acc_original) >= 0)

# Rationale: a model trained under one distribution can degrade under shift; monitor feature drift + performance drift.

In [ ]:
# Checks
check('shift_changes_acc', abs(acc_shift - acc_original) >= 0)

**Why this works.** a model trained under one distribution can degrade under shift; monitor feature drift + performance drift.

## Task 6: Show that accuracy is misleading under imbalance
*Section: Class Imbalance Trap*

## Section 5 — Class Imbalance Trap

### Task 5.1: Show that accuracy is misleading under imbalance

Downsample positives to ~5%, train a model, and show accuracy looks great but recall is terrible.

**FAANG gotcha:** "My model has 95% accuracy" means nothing if 95% of labels are negative.

In [ ]:
# Class imbalance demo — use synthetic data with 95% negatives
from sklearn.datasets import make_classification
X_imb, y_imb = make_classification(n_samples=200, n_features=5, n_informative=3,
    n_redundant=1, weights=[0.95, 0.05], flip_y=0.03, random_state=0)
print("class distribution:", np.bincount(y_imb))

Xtr_i, Xva_i, ytr_i, yva_i = train_test_split(X_imb, y_imb, test_size=0.3, random_state=0, stratify=y_imb)
clf_imb = LogisticRegression(max_iter=2000)
clf_imb.fit(Xtr_i, ytr_i)
pred_imb = clf_imb.predict(Xva_i)

acc_imb = accuracy_score(yva_i, pred_imb)
prec, rec, f1, _ = precision_recall_fscore_support(yva_i, pred_imb, pos_label=1, average="binary", zero_division=0)
print("acc_imbalanced", acc_imb)
print("precision", prec, "recall", rec, "f1", f1)
check("acc_misleading_high", acc_imb > 0.8)
check("recall_reveals_problem", rec < 0.8)

# Rationale: under class imbalance, accuracy rewards predicting the majority class; always check precision/recall/F1.

In [ ]:
# Checks
check("acc_misleading_high", acc_imb > 0.8)
check("recall_reveals_problem", rec < 0.8)

**Why this works.** under class imbalance, accuracy rewards predicting the majority class; always check precision/recall/F1.

## Section 5 — Class Imbalance Trap

### Task 5.1: Show that accuracy is misleading under imbalance

Downsample positives to ~5%, train a model, and show accuracy looks great but recall is terrible.

**FAANG gotcha:** "My model has 95% accuracy" means nothing if 95% of labels are negative.